<a href="https://colab.research.google.com/github/Telop-Auto/telopauto/blob/v1.6/Telop_Auto_v1_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telop Auto V1.6
Whisper(音声認識)と Gemini(校正)で、音声ファイルから字幕(SRT)を自動作成します。

制作・著作:Telop Design Lab ／ 不具合や改善点は X の DM までご連絡ください:https://x.com/telopdesignlab

作成した字幕は、Telop Design Lab のテロップテンプレートで Premiere 上でデザインを適用できます:https://terop.booth.pm/

## 使い方
1. 上部メニュー「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選ぶ
2. 下のセルのフォームで、行数・文字数などを設定する
3. セルの **▶** を押す → GPU と Gemini APIキーを確認したあと、ファイル選択ボタンが出るので音声ファイルを選ぶ
4. あとは最後まで自動で進み、SRT がダウンロードされます

**作り直したいとき**は、設定を変えて「use_previous_file」にチェックを入れ、もう一度 ▶ を押してください。アップロードは不要で、変更が必要な工程だけやり直します。エラーで止まった場合も同じ操作で続きから再開できます。

<details><summary>🔑 Gemini APIキーの設定(初回のみ・クリックで開く)</summary>

Gemini 校正を使う場合は、APIキーの登録が必要です(Google AI Studio で無料で取得できます)。

1. 左側の 🔑(シークレット)を開く
2. 「新しいシークレットを追加」で、名前に `GEMINI_API_KEY`、値に APIキーを入力
3. 「ノートブックからのアクセス」をオンにする

Gemini 校正を使わない場合は、フォームの「use_gemini」のチェックを外してください。
</details>

In [ ]:
# @title 🎬 Telop Auto　設定して ▶ を押してください { display-mode: "form" }

# @markdown ## 字幕の形
# @markdown 1行の文字数の目安(推奨:20)
max_chars_per_line = 20  # @param {type:"integer"}
# @markdown 字幕の行数(推奨:2行)
subtitle_lines = "2行"  # @param ["1行", "2行"]
# @markdown 1枚(1ブロック)の最大文字数(2行字幕なら「1行の文字数×2」が目安)
block_max_chars = 40  # @param {type:"integer"}

# @markdown ---
# @markdown ## フィラー削除・校正
# @markdown フィラー削除の強さ(推奨:中)
filler_level = "中"  # @param ["弱", "中", "強"]
# @markdown Gemini で誤字・誤変換を校正する(APIキーが必要)
use_gemini = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ## 作り直すとき
# @markdown チェックすると、アップロードを飛ばして前回の音声ファイルで作り直します(変更が必要な工程だけ実行)
use_previous_file = False  # @param {type:"boolean"}
# @markdown チェックすると、文字起こしから全部やり直します
start_over = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ## 詳細設定(通常は変更不要)
# @markdown Whisperモデル(large-v3:高精度 ／ medium:高速・長尺向け)
whisper_model = "large-v3"  # @param ["large-v3", "medium"]
# @markdown 追加で削除するフィラー(カンマ区切り)
custom_fillers = "あのー,えっと,ええと,なんか"  # @param {type:"string"}
# @markdown ユーザー辞書「置換前=置換後」(複数は | で区切る)。登録した語は途中で改行されません
user_dictionary = "プレミア=Premiere | アイフォン=iPhone | 厳選かけ流し=源泉かけ流し"  # @param {type:"string"}
# @markdown Gemini補助辞書:校正時に参考にする固有名詞(カンマ区切り)。登録した語は途中で改行されません
gemini_assist_words = "iPhone,Premiere,Telop Design Lab"  # @param {type:"string"}
# @markdown Geminiモデル名
gemini_model = "gemini-3.5-flash"  # @param {type:"string"}

# ==========================================
# 以下は編集不要
# ==========================================
TELOPAUTO_REF = "v1.6"


def _telopauto_main():
    import importlib.util
    import subprocess
    import sys

    # GPU の確認(インストールより先に行い、設定ミスなら待たせずに止める)
    import torch

    if not torch.cuda.is_available():
        print("⚠️ GPU が使えない状態です。")
        print("上部メニュー「ランタイム」→「ランタイムのタイプを変更」で「T4 GPU」を選んで保存し、")
        print("もう一度 ▶ を押してください(アップロードはまだ不要です)。")
        return

    # 初回だけインストール
    marker = "/content/.telopauto_ref"
    installed = None
    try:
        installed = open(marker).read().strip()
    except OSError:
        pass
    if importlib.util.find_spec("telopauto") is None or installed != TELOPAUTO_REF:
        print("=== 初回の準備をしています(1〜2分) ===")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             f"git+https://github.com/Telop-Auto/telopauto@{TELOPAUTO_REF}"],
            check=True,
        )
        with open(marker, "w") as f:
            f.write(TELOPAUTO_REF)
        print("準備完了\n")

    from telopauto import app
    from telopauto.settings import Settings

    app.run(
        Settings(
            max_chars_per_line=max_chars_per_line,
            lines=1 if subtitle_lines == "1行" else 2,
            block_max_chars=block_max_chars,
            whisper_model=whisper_model,
            filler_level=filler_level,
            custom_fillers=custom_fillers,
            user_dictionary=user_dictionary,
            use_gemini=use_gemini,
            gemini_model=gemini_model,
            gemini_assist_words=gemini_assist_words,
        ),
        use_previous_file=use_previous_file,
        start_over=start_over,
    )


_telopauto_main()

## 注意事項
- **長尺素材は MP3 を推奨します**(WAV はアップロードに時間がかかることがあります)。動画ファイル(MP4 など)もそのまま使えます。
- 本ツールは継続的に改善・アップデートを行っているβ版です。文字起こしや校正の結果には誤りが含まれる場合があります。最終的な内容は必ず確認してください。
- Gemini API には利用制限があります。上限に達した場合、残りの文は校正せずに SRT を作成します。時間をおいて「use_previous_file」にチェックを入れて ▶ を押すと、未校正の文だけを校正します。
- Google Colab の無料 GPU には利用上限があります。
- 本ノートブックは個人・業務用途を問わずご利用いただけます。ただし、本ノートブックおよびその複製・改変物の再配布・再販売は禁止します。